# YOLO 모델 평가

- 학습된 모델 평가
- 원본 스크립트: `eval_yolo_cls.py`

YOLO Classification 평가 스크립트
학습된 YOLO 분류 모델을 평가합니다.

사용법:
    python scripts/eval_yolo_cls.py --model runs/classify/fine_cls_v2/weights/best.pt --mode fine

In [14]:
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any, Dict

import pandas as pd
from IPython.display import display

# conda activate archlens
# pip install ultralytics
from ultralytics import YOLO

In [16]:
# ----- Helpers -----

def ensure_file(path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError(f"파일이 없습니다: {path}")
    return path


def ensure_dir(path: Path) -> Path:
    if not path.is_dir():
        raise FileNotFoundError(f"디렉터리가 없습니다: {path}")
    return path


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="YOLO-CLS 평가 설정")
    # 모델을 지정하지 않으면 최신 yolo-cls*의 best.pt로 fallback (config 셀에서 처리)
    parser.add_argument("--model", type=Path, default=None)
    parser.add_argument("--mode", choices=["fine", "coarse"], default="fine")
    parser.add_argument("--data-dir", type=Path, default=Path("./dataset/icons"))
    parser.add_argument("--split", choices=["val", "test"], default="val")
    parser.add_argument("--save-json", action="store_true", help="metrics를 JSON으로 저장")
    parser.add_argument("--name", default="yolo-cls-eval", help="runs/classify 하위 실험 이름")
    return parser.parse_args(args=[])


# ----- Config -----
args = parse_args()
MODE: str = args.mode
DATA_DIR: Path = ensure_dir(args.data_dir)
YOLO_DATA_DIR: Path = ensure_dir(DATA_DIR / f"yolo_cls_{MODE}")
SPLIT: str = args.split
SAVE_JSON: bool = args.save_json

# 최신 yolo-cls* run에서 자동 fallback하여 best.pt 찾기
import re

def find_latest_run(base: Path) -> Path:
    numbered = []
    for p in base.iterdir():
        if not p.is_dir():
            continue
        m = re.match(r"^yolo-cls(\d+)$", p.name)
        if m:
            numbered.append((int(m.group(1)), p))
    if numbered:
        return max(numbered, key=lambda t: t[0])[1]
    candidates = sorted(
        [p for p in base.iterdir() if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else base / "yolo-cls"

base_runs = Path("runs/classify").resolve()
latest_run = find_latest_run(base_runs)

# args.model이 주어지면 그대로, 아니면 최신 run의 best.pt
if args.model:
    MODEL_PATH: Path = ensure_file(args.model)
else:
    MODEL_PATH = ensure_file(latest_run / "weights" / "best.pt")

RUN_NAME: str = args.name if args.name else latest_run.name

print(f"[INFO] MODEL    = {MODEL_PATH}")
print(f"[INFO] MODE     = {MODE}")
print(f"[INFO] DATA     = {YOLO_DATA_DIR} (split={SPLIT})")
print(f"[INFO] RUN NAME = {RUN_NAME} (latest: {latest_run.name})")
print(f"[INFO] SAVE JSON= {SAVE_JSON}")

[INFO] MODEL    = /home/wsm/workspace/hit-archlens-project/runs/classify/yolo-cls16/weights/best.pt
[INFO] MODE     = fine
[INFO] DATA     = dataset/icons/yolo_cls_fine (split=val)
[INFO] RUN NAME = yolo-cls-eval (latest: yolo-cls16)
[INFO] SAVE JSON= False


In [18]:
def evaluate() -> Any:
    """Ultralytics 분류 모델 평가를 수행하고 결과를 반환한다."""

    model = YOLO(str(MODEL_PATH))
    results = model.val(
        data=str(YOLO_DATA_DIR),
        split=SPLIT,
        project="runs/classify",
        name=RUN_NAME,
        task="classify",
    )
    return results


def summarize(results: Any) -> Dict[str, Any]:
    # ultralytics classify의 val() 반환이 ClassifyMetrics 자체일 수 있으므로 직접 참조
    top1 = getattr(results, "top1", None)
    top5 = getattr(results, "top5", None)
    loss = getattr(results, "loss", None)
    summary = {
        "split": SPLIT,
        "top1": top1,
        "top5": top5,
        "loss": loss,
    }
    display(pd.DataFrame([summary]))
    if SAVE_JSON:
        out_dir = Path("runs/classify") / RUN_NAME
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / "metrics.json"
        out_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
        print(f"[INFO] metrics saved -> {out_path}")
    return summary


results = evaluate()
metrics_summary = summarize(results)
metrics_summary

Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32108MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,532,032 parameters, 0 gradients, 4.3 GFLOPs
train: /home/wsm/workspace/hit-archlens-project/dataset/icons/yolo_cls_fine/train... found 447 images in 64 classes ✅ 
val: /home/wsm/workspace/hit-archlens-project/dataset/icons/yolo_cls_fine/val... found 96 images in 64 classes ✅ 
test: /home/wsm/workspace/hit-archlens-project/dataset/icons/yolo_cls_fine/test... found 96 images in 64 classes ✅ 
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 974.0±634.3 MB/s, size: 5.1 KB)
val: Scanning /home/wsm/workspace/hit-archlens-project/dataset/icons/yolo_cls_fine/val... 96 images, 0 corrupt: 100% ━━━━━━━━━━━━ 96/96 534.7Kit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 6/6 46.8it/s 0.1s.4s
                   all       0.25       0.75
Speed: 0.2ms preprocess, 0.5ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /hom

,split,top1,top5,loss
0,val,0.25,0.75,None


{'split': 'val', 'top1': 0.25, 'top5': 0.75, 'loss': None}